# Imports

In [ ]:
from Functions.Fcts_Filtering import (filter_organoids_by, calculate_outgrowth, remove_cut_organoids,
                                      find_outlier_organoids, plot_random_organoids)
from Functions.Fcts_Base import save_after_filtering, load_adata
import seaborn as sns


%load_ext autoreload
%autoreload 2

# User Input

In [ ]:
load_dir = "PATH TO ANNDATA OBJECT FROM 1_FeatureExtraction" # Path to .h5ad file containing the AnnData object.
condition_cols = ["Medium", "Cell_line", "ABs", "Other"] # Columns in adata.obs that together define a condition (used by the outlier check).

pyramid_lvl_visualization = 1 # Used for visualization of images in plots. 0: full resolution, 1: downsampled by factor 2, etc.

# Set up plotting parameters
fig_param = {"axes.spines.right": False,
                  "axes.spines.top": False,
                  "axes.spines.left": True,
                  "axes.spines.bottom": True,
                  "pdf.fonttype": 42,
                  "font.size": 6,
                  "axes.labelsize": 6,
                  "axes.titlesize": 8,
                  "xtick.labelsize": 6,
                  "ytick.labelsize": 6,
                  "legend.fontsize": 6}

sns.set_theme(rc = fig_param, style = "ticks")

# Load

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad_raw = load_adata(load_dir, load_zarrs = True)

# Filter debris
Removes objects outside a range of a feature that are clearly not organoids (debris, empty or failed segmentations) and visualizes the removed objects. Use boundaries only for clear artifacts: an upper size boundary, for example, can remove a real phenotype (large organoids). Rare unusual objects are found by the outlier check below, within each condition. Every removal is recorded with its reason in ad.uns["removed_organoids"]. The first filter starts from ad_raw, so re-running from there starts over.

### Low DAPI brightness

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
feat = "R0__DAPI_mean"          # Name of the numerical feature to filter.
lower_threshold = 50            # Lower boundary. None: no lower boundary.
upper_threshold = 3000          # Upper boundary. None: no upper boundary.
channel_visualization = "R0__DAPI" # Channel used for visualization of images in plots.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = filter_organoids_by(ad_raw,
                    feature = feat,
                    values = (lower_threshold, upper_threshold),
                    channel = channel_visualization,
                    pyramid_level = pyramid_lvl_visualization
                    )

### Low size

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
feat = "area"               # Name of the numerical feature to filter.
lower_threshold = 100       # Lower boundary. None: no lower boundary.
upper_threshold = None      # Upper boundary. None: no upper boundary (large organoids can be a real phenotype; merged organoids are found by the outlier check).
channel_visualization = "R0__DAPI" # Channel used for visualization of images in plots.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = filter_organoids_by(ad,
                    feature = feat,
                    values = (lower_threshold, upper_threshold),
                    channel = channel_visualization,
                    pyramid_level = pyramid_lvl_visualization,
                    add_boundary = False
                    )

# Calculate outgrowth
Calculates the outgrowth information per well by taking all indicated wells in the .xls setup file into account, even if no organoids are detected during the segmentation. Runs after the debris filters, so debris does not count. Organoids removed below as cut off still count, since they are organoids that are only not completely imaged. The plate QC in 3_QualityControl uses these counts.

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
n_seeded = 175 # Number of cells/structures seeded at day 0 per well. Assumed to be constant.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = calculate_outgrowth(ad = ad, n_seeded = n_seeded)

# Cut-off organoids
Removes organoids whose bounding box touches the border of the well image: they are only partly imaged, so their size, shape and intensities are wrong. The image size per plate is stored in ad.uns["image_size"] and reused in 3_QualityControl. Without bounding box columns, filter on StraightEdge_longest with filter_organoids_by instead.

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
border_tolerance = 1.0 # Distance (µm) to the image border that still counts as touching.
channel_visualization = "R0__DAPI" # Channel used for visualization of images in plots.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = remove_cut_organoids(ad,
                    border_tolerance = border_tolerance,
                    channel = channel_visualization,
                    pyramid_level = 0,
                    add_boundary = False
                    )

# Outlier organoids
Finds organoids unlike any other organoid of their condition (likely merged organoids, debris or segmentation errors), using the local outlier factor on rank-based scores, and visualizes them. Heterogeneous but common phenotypes are not flagged. Runs after the cut-off organoids are removed, since these are common and would form their own group.

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
outlier_threshold = 2.0 # Local outlier factor from which an organoid is an outlier (1 = as dense as its neighbors, 2 = half as dense).
remove_outliers = True # True: remove the outlier organoids. False: only mark them in ad.obs["organoid_outlier"].
channel_visualization = "R0__DAPI" # Channel used for visualization of images in plots.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
ad = find_outlier_organoids(
    ad,
    condition_cols = condition_cols,
    threshold = outlier_threshold,
    remove_outliers = remove_outliers,
    channel = channel_visualization,
    pyramid_level = pyramid_lvl_visualization)

# Sample of remaining objects

In [ ]:
# ── Function Parameters ───────────────────────────────────────────────────────────
feature = "area" #Feature to display in title below unique ID
channel_visualization = "R0__DAPI" # Channel used for visualization of images in plots.

# ── Run Cell ───────────────────────────────────────────────────────────────────────
fig = plot_random_organoids(ad,
                            feature = feature,
                            rows = 10,
                            cols = 10,
                            channel = channel_visualization,
                            seed = 42,
                            pyramid_level = pyramid_lvl_visualization,
                            add_boundary = False)

# Save DF/AD

In [ ]:
# ── Run Cell ───────────────────────────────────────────────────────────────────────
save_after_filtering(ad)